# NICTO AI Training
Train the NICTO model on Google Colab (Free T4 GPU).

**Steps:**
1. Runtime -> Change runtime type -> T4 GPU
2. Run all cells top to bottom
3. Training starts automatically

Model: ~100M params (fits on T4 with room to spare)
Time: ~30 min for 2000 steps on synthetic data
For real training: upload a text file and set DATA_PATH below

In [ ]:
# Install dependencies
!pip install torch --quiet
!pip install datasets --quiet

import torch
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB" if torch.cuda.is_available() else "No GPU")

In [ ]:
# Clone NICTO repo (or upload files manually)
import os
REPO_PATH = "/content/NICTO"

if not os.path.exists(REPO_PATH):
    # Option 1: Clone from GitHub (if you have a repo)
    # !git clone https://github.com/YOUR_USERNAME/NICTO.git
    
    # Option 2: Upload files manually
    # Upload the nicto_ai/ folder using the file browser on the left
    print("Upload the NICTO folder to /content/ using the file browser")
    print("Or connect your GitHub repo above")
else:
    print("NICTO repo found!")
    os.chdir(REPO_PATH)
    print(f"Working directory: {os.getcwd()}")

In [ ]:
# Verify installation
import sys
sys.path.insert(0, "/content")
sys.path.insert(0, "/content/NICTO")

from nicto_ai.training.model_train import NICTOTrainModel, NICTOTrainConfig

config = NICTOTrainConfig()
model = NICTOTrainModel(config)
params = model.count_parameters()
print(f"Model created: {params:,} parameters ({params/1e6:.0f}M)")
print(f"Config: dim={config.dim}, heads={config.n_heads}, experts={config.moe_experts}")

## Option A: Train on Synthetic Data (Quick Test)
This tests that the training pipeline works. Takes ~5 minutes.

In [ ]:
# Quick test: 200 steps on synthetic data
from nicto_ai.training.train import train

# Override config for quick test
import nicto_ai.training.train as trainer

model = train("colab")

## Option B: Train on Real Data
Upload a text file or dataset for actual learning.

In [ ]:
# Upload your training data
# Option 1: Upload a .txt file using the file browser
# Option 2: Download a public dataset

DATA_PATH = None  # Set to your file path, e.g., "/content/training_data.txt"

# Example: Download OpenWebText sample
# from datasets import load_dataset
# ds = load_dataset("openwebtext", split="train", streaming=True)
# with open("/content/training_data.txt", "w") as f:
#     for i, sample in enumerate(ds):
#         if i >= 10000: break
#         f.write(sample["text"] + "\n")
# DATA_PATH = "/content/training_data.txt"

if DATA_PATH:
    print(f"Training data: {DATA_PATH}")
else:
    print("No data set. Using synthetic data.")
    print("Upload a .txt file and set DATA_PATH above for real training.")

## Download Trained Model
After training, download your checkpoint.

In [ ]:
# List checkpoints
import os
from pathlib import Path

ckpt_dir = Path("nicto_ai/training/checkpoints/colab")
if ckpt_dir.exists():
    for f in sorted(ckpt_dir.glob("*.pt")):
        size_mb = f.stat().st_size / 1024 / 1024
        print(f"  {f.name}: {size_mb:.1f} MB")
    
    # Download final model
    final = ckpt_dir / "final.pt"
    if final.exists():
        from google.colab import files
        files.download(str(final))
        print("Downloading final.pt...")
else:
    print("No checkpoints found. Run training first.")